## 0. Ayarlar
`kx.py new` bu hucredeki alanlari doldurur. MODE ve hiperparametreler elle ayarlanir.

In [ ]:
EXP_ID  = "__EXP_ID__"
PARENT  = "__PARENT__"
OWNER   = "__OWNER__"
SLUG    = "__SLUG__"
NOTE    = "__NOTE__"

MODE = "FAST"          # FAST | FULL  -> FAST tanimi core/cv_spec.md'de SABITTIR
SEED = 42

import os, sys, json, time, warnings, platform
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
np.random.seed(SEED)

T0 = time.time()
WORK = "/kaggle/working"
ART  = os.path.join(WORK, "artifacts")
os.makedirs(ART, exist_ok=True)
print(EXP_ID, "|", MODE, "| seed", SEED, "| parent", PARENT)

## 1. Yol dogrulamasi
**Sessizce devam etme.** Beklenen dosya adi ve semasi kontrol edilir; birden fazla
eslesmede dur. Yanlis dosyayi okuyan bir notebook 3 saat sonra fark edilir.

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
print("Girdiler (ham agac):")
for p in sorted(INPUT.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(INPUT))

def bul(required_files, hint):
    """/kaggle/input altinda required_files'in TAMAMINI iceren ilk klasoru bulur.
    Kaggle'in girdi yerlesimi SABIT DEGIL: interaktif editorde duz
    /kaggle/input/<slug>/ kullanilirken, `kaggle kernels push` ile (CLI/batch)
    baslatilan kosularda ic ice /kaggle/input/competitions/<slug>/ ve
    /kaggle/input/datasets/<owner>/<slug>/ yapisi gorulebiliyor (provada
    yakalandi - EXP-001, 22 Eyl). Bu yuzden sabit yol yerine arama yapilir."""
    for p in sorted(INPUT.rglob("*")):
        if p.is_dir() and all((p / f).exists() for f in required_files):
            return p
    raise FileNotFoundError(
        f"{hint}: {required_files} iceren klasor /kaggle/input altinda yok. "
        "Ust hucredeki agaca bak, dataset_sources/competition_sources ekli mi kontrol et."
    )

COMP = bul(["train.csv", "test.csv", "sample_submission.csv"], "yarisma verisi")
CORE = bul(["folds.csv", "metric.py"], "hackathon-core")
print("COMP:", COMP)
print("CORE:", CORE)
print("CORE:", CORE)

## 2. Fold ve metrik — `hackathon-core`'dan
Notebook **kendi fold'unu uretmez.** Metrik de burada tanimlanmaz.
Ikisi de tek kaynaktan okunur; yoksa 6 kosunun skoru karsilastirilamaz.

In [ ]:
sys.path.insert(0, str(CORE))
from metric import score, GREATER_IS_BETTER      # core/metric.py

folds = pd.read_csv(CORE / "folds.csv")
assert "fold" in folds.columns, "folds.csv icinde 'fold' kolonu yok"
ID = folds.columns[0]
N_FOLDS = folds["fold"].nunique()
print("fold sayisi:", N_FOLDS, "| satir:", len(folds), "| id kolonu:", ID)

## 3. Veri

In [ ]:
train = pd.read_csv(COMP / "train.csv")     # TODO: gercek dosya adi
test  = pd.read_csv(COMP / "test.csv")      # TODO
sample = pd.read_csv(COMP / "sample_submission.csv")  # TODO

TARGET = "TODO"
assert TARGET in train.columns

# SADECE fold birlestir - folds.csv'nin ekstra kolonlari (orn. y) kazara
# ozellik listesine sizmasin (Codex kod incelemesi, EXP-001, 22 Eyl: tam
# birlestirme hedefi FEATURES'a sizdirdi, hem sizinti hem test'te KeyError).
train = train.merge(folds[[ID, "fold"]], on=ID, how="left")
assert train["fold"].notna().all(), "folds.csv train ile tam eslesmiyor"
print(train.shape, test.shape)

## 4. Ozellik hazirligi
**Fit edilen her donusum fold ICINDE fit edilir.** Buraya sadece fold'dan bagimsiz,
satir-bazli donusumler girer (tip cevrimi, basit ayristirma). Target encoding,
scaler, imputer gibi her sey asagidaki dongunun icindedir.

In [ ]:
FEATURES = [c for c in train.columns if c not in (ID, TARGET, "fold")]
print(len(FEATURES), "ozellik")

## 5. Egitim dongusu
Her fold bitince: skor basilir **ve ara cikti diske yazilir.**
Sure limitine takilirsa yarisi kurtulur.

In [ ]:
def make_model(seed):
    # TODO: model. Ornek:
    # from sklearn.ensemble import HistGradientBoostingClassifier
    # return HistGradientBoostingClassifier(random_state=seed)
    raise NotImplementedError("model tanimla")

oof_pred = np.full(len(train), np.nan, dtype=float)
test_parts, fold_scores = [], []
fold_list = sorted(train["fold"].unique())
if MODE == "FAST":
    fold_list = fold_list[:2]          # FAST tanimi cv_spec.md'de sabit

model = None
for f in fold_list:
    tr = train[train["fold"] != f]
    va = train[train["fold"] == f]

    # --- fold ICINDE fit edilen her sey buraya
    model = make_model(SEED)
    model.fit(tr[FEATURES], tr[TARGET])

    p = model.predict(va[FEATURES])
    oof_pred[va.index] = p
    test_parts.append(model.predict(test[FEATURES]))

    s = score(va[TARGET].values, p)
    fold_scores.append(float(s))
    print(f"fold {f}: {s:.6f}  ({time.time()-T0:.0f}s)")

    # ara cikti — sure limiti panzehiri
    pd.DataFrame({ID: train[ID], "pred": oof_pred}).to_parquet(f"{WORK}/oof_partial.parquet", index=False)

print("fold skorlari:", fold_scores)

## 6. Cikti sozlesmesi
`core/cv_spec.md` Bolum "Cikti sozlesmesi". Kimliksiz `.npy` yasak.
`kx.py fetch` bunlari otomatik dogrular; tutmazsa sonuc kayda girmez.

In [ ]:
done = train["fold"].isin(fold_list)

oof = pd.DataFrame({ID: train.loc[done, ID].values, "pred": oof_pred[done.values]})
oof.to_parquet(f"{WORK}/oof.parquet", index=False)

test_pred = np.mean(test_parts, axis=0)
test_pred_by_id = pd.Series(test_pred, index=test[ID].values)
pd.DataFrame({ID: test[ID].values, "pred": test_pred}).to_parquet(f"{WORK}/test_preds.parquet", index=False)

# ID'ye gore hizala - pozisyonel atama DEGIL. test ve sample_submission'in
# satir sirasi ayni garantisi yoktur; farkliysa pozisyonel atama sessizce
# yanlis kisiye tahmin yapistirir (Codex kod incelemesi, EXP-001, 22 Eyl).
sub = sample.copy()
aligned = test_pred_by_id.reindex(sub[ID].values)
assert aligned.notna().all(), "submission id'lerinden bazilari test tahmininde yok"
sub[sub.columns[1]] = aligned.values          # TODO: cok kolonluysa duzelt
assert len(sub) == len(sample) and list(sub.columns) == list(sample.columns)
sub.to_csv(f"{WORK}/submission.csv", index=False)

cv_mean = float(np.mean(fold_scores))
cv_oof  = float(score(train.loc[done, TARGET].values, oof_pred[done.values]))

result = {
    "exp_id": EXP_ID, "parent": PARENT, "owner": OWNER,
    "kernel_slug": SLUG, "kernel_version": os.environ.get("KAGGLE_KERNEL_RUN_VERSION", "?"),
    "mode": MODE, "seed": SEED,
    "fold_scores": fold_scores, "cv_mean": cv_mean, "cv_oof": cv_oof,
    "n_folds_done": len(fold_scores), "n_rows_oof": int(len(oof)),
    "runtime_min": round((time.time()-T0)/60, 1), "note": NOTE,
}
json.dump(result, open(f"{WORK}/result.json", "w"), indent=2)
print(json.dumps(result, indent=2))

## 7. Artefaktlar
**Ikinci asama sigortasi.** Bu hucre 10 saniye surer ve Cumartesi ogleni kurtarir:
KABUL alan bir deneyin modelini yeniden egitmek zorunda kalmazsin.

In [ ]:
import pickle
pickle.dump(model, open(f"{ART}/model.pkl", "wb"))
json.dump(
    {"features": FEATURES, "target": TARGET, "id_col": ID, "seed": SEED,
     "classes": getattr(model, "classes_", None).tolist() if hasattr(model, "classes_") else None,
     "python": platform.python_version(),
     "packages": {"numpy": np.__version__, "pandas": pd.__version__}},
    open(f"{ART}/meta.json", "w"), indent=2)
print("artifacts:", os.listdir(ART))